In [ ]:
import geopandas as gpd
from matplotlib import pyplot as plt
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import sys
import os

sys.path.append(os.path.abspath(".."))
from utils.plot_style import apply_plot_style
from utils.config import root_dir

apply_plot_style()

In [ ]:
# load data layers
us_states_gdf = gpd.read_file(
     f"zip://{root_dir}/data_input/state_boundaries/cb_2018_us_state_20m.zip/cb_2018_us_state_20m.shp"
)
aoi_counties = gpd.read_file(root_dir + "results/aoi_county_boundaries.gpkg")
aoi_regions = aoi_counties[["Region", "geometry"]].dissolve(by="Region").reset_index()
aoi_huc12s = gpd.read_file(root_dir + "results/aoi_huc12_boundaries.gpkg")

In [ ]:
#  Configuration & Data Setup
color_dict = {
    "Headwaters": "#0387d1",
    "Gorge": "#fbbc04",
    "Driftless": "#0a7138",
    "Working River": "#7030a0",
    "Confluence": "#e65100",
    "Chickasaw": "#1a237e",
    "Delta": "#87ceac",
    "Lower Mississippi": "#8a0a55",
    "Gulf South": "#ffff8c",
}
geography_labels = {
    "Headwaters": "Lake Itasca to Little Falls, MN",
    "Gorge": "Little Falls, MN to Bay City, WI",
    "Driftless": "Bay City, WI to Clinton, IA",
    "Working River": "Clinton, IA to Elsberry, MO",
    "Confluence": "Elsberry, MO to Hickman, KY",
    "Chickasaw": "Hickman, KY to Randolf, TN",
    "Delta": "Randolf, TN to Vicksburg, MS",
    "Lower Mississippi": "Vicksburg, MS to St. Francisville, LA",
    "Gulf South": "St. Francisville, LA to Gulf of Mexico",
}

# Apply color mapping
aoi_huc12s["color_col"] = aoi_huc12s["Region"].map(color_dict)

In [ ]:
# Fixed figure size in inches; this becomes the PDF size on export.
fig, ax = plt.subplots(1, 1, figsize=(3.42, 4.8))

# Plot Layers
# Background states
us_states_gdf.to_crs(aoi_huc12s.crs).plot(
    ax=ax, edgecolor="#adb5bd", facecolor="#f8f9fa", lw=1.0
)
# HUC12 Regions with custom colors
aoi_huc12s.plot(ax=ax, color=aoi_huc12s["color_col"], alpha=0.9)
# Regional county boundaries
aoi_regions_projected = aoi_regions.to_crs(aoi_huc12s.crs)
aoi_regions_projected.plot(
    ax=ax,
    edgecolor="black",
    facecolor="none",
    lw=0.8,
    joinstyle="round",
    capstyle="round",
)

# Set Map Extent (Locking the view to the river area but with some margins)
minx, miny, maxx, maxy = aoi_huc12s.total_bounds
ax.set_xlim(minx - 50000, maxx + 50000)
ax.set_ylim(miny - 50000, maxy + 50000)
ax.set_xticks([])
ax.set_yticks([])

# Add Labels and Callout Lines
# label_x_anchor: 1.02 means 2% outside the right edge of the map axis
label_x_anchor = 1.02

for _, row in aoi_regions_projected.iterrows():
    region_name = row["Region"]
    # Get a point inside the polygon
    coords = row["geometry"].representative_point().coords[0]

    # Geography Callout OUTSIDE the map
    if region_name in geography_labels:
        geo_text = geography_labels[region_name]
        n = len(aoi_huc12s[aoi_huc12s["Region"] == region_name])
        # We combine Region and Geo text.
        # Using ax.get_yaxis_transform() allows us to pin the X position
        # to the axis edge while the Y position follows the map data.
        bold_name = r"$\bf{" + region_name.replace(" ", r"\ ") + r"}$"
        full_label = f"{bold_name}, n = {n}\n{geo_text}"
        # Thick white arrow to act as halo
        ax.annotate(
            text="",
            xy=coords,
            xytext=(label_x_anchor, coords[1]),
            textcoords=ax.get_yaxis_transform(),
            arrowprops={
                "arrowstyle": "<-",
                "color": "white",
                "linewidth": 3,
                "shrinkA": 5,
                "connectionstyle": "arc3,rad=0",
            },
        )
        ax.annotate(
            text=full_label,
            xy=coords,
            xytext=(label_x_anchor, coords[1]),
            textcoords=ax.get_yaxis_transform(),
            arrowprops={
                "arrowstyle": "<-",
                "color": "black",
                "linewidth": 1,
                "shrinkA": 5,
                "connectionstyle": "arc3,rad=0",
            },
            va="center",
            ha="left",
        )

plt.axis("off")

# add a scale bar at the bottom in km
scalebar = AnchoredSizeBar(
    ax.transData,
    size=200000,  # Length of bar in data coordinates (meters)
    label="200 km",  # Label text
    loc="lower left",  # Position
    pad=0.5,
    borderpad=0.5,
    color="black",
    frameon=False,  # No background box
    size_vertical=2000,  # Height/thickness of bar in data units
)

ax.add_artist(scalebar)

# Final Layout Adjustment
# This shrinks the map area to 60% of the figure width,
# leaving 40% on the right for your labels.
plt.subplots_adjust(right=0.47, left=0.01, top=0.99, bottom=0.01)

plt.show()

fig.savefig(f"{root_dir}/figures/Figure 1.pdf")